# Land-Use Data Cleaning: CDL HUC-12 Cover Fractions

Cleans the **per-watershed land-cover composition** table derived from the USDA
Cropland Data Layer (CDL) into a tidy, type-safe table keyed on
`huc12_code` + `year`, ready to join as a watershed-scale land-use covariate.

**Input:**  `data/tabular/01_raw/landuse/cdl-huc12-fractions.csv`
**Output:** `data/tabular/02_clean/landuse/cdl-huc12-fractions-clean.csv`

Each raw row gives, for one HUC-12 watershed in one `year` (2015–2025), the
fraction of land area in each cover class. The eight mutually-exclusive cover
classes (`pct_corn`, `pct_soybean`, `pct_other_crops`, `pct_developed`,
`pct_forest`, `pct_pasture`, `pct_wetland`, `pct_open_water`) sum to ≈1;
`pct_row_crops` is a *derived* aggregate (corn + soybean + other row crops) and
overlaps those classes, so it is **not** part of the sum.

**Cleaning steps:**

1. Load the raw extract (the HUC id as a string).
2. **Fix the join key** — zero-pad `HUC_12` to the canonical 12-digit string
   (reading it as an integer silently dropped the leading `0` from the ~13k
   codes in HUC region `07`) and rename it `huc12_code` to match the snake-case
   id convention used by the other cleaners (e.g. `huc8_code`).
3. **Range-validate** the fractions — every `pct_*` must lie in `[0, 1]`.
4. **De-duplicate** on `(huc12_code, year)` so the key is unique.
5. **Sort & write** the tidy table to `02_clean`.

> **Path note:** like the other migrated cleaners this reads `01_raw` and writes
> `02_clean`. No land-use merge step exists yet; when one is added it should read
> from `02_clean/landuse/` and join on the zero-padded `huc12_code`.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies, so
    resolving paths relative to a fixed number of "../" is fragile. Searching
    upward for a sentinel makes the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "landuse"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "landuse"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Raw dir:  ", RAW_DIR)
print("Clean dir:", CLEAN_DIR)

Repo root: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction
Raw dir:   /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/01_raw/landuse
Clean dir: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/landuse


## Step 1 — Load

~18.9K rows (1,714 watersheds × 11 years). We read `HUC_12` as a string up
front: it is a categorical identifier (never arithmetic, and leading-zero
sensitive). The `pct_*` columns are genuine numerics and read as floats.

In [3]:
df = pd.read_csv(
    RAW_DIR / "cdl-huc12-fractions.csv",
    dtype={"HUC_12": "string"},
)
n_raw = len(df)
PCT_COLS = [c for c in df.columns if c.startswith("pct_")]
print(f"Loaded {n_raw:,} rows")
print("Columns:", list(df.columns))
print("Years:  ", sorted(df["year"].unique()))
df.head()

Loaded 18,854 rows
Columns: ['year', 'HUC_12', 'pct_corn', 'pct_soybean', 'pct_other_crops', 'pct_developed', 'pct_forest', 'pct_pasture', 'pct_wetland', 'pct_open_water', 'pct_row_crops']
Years:   [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,year,HUC_12,pct_corn,pct_soybean,pct_other_crops,pct_developed,pct_forest,pct_pasture,pct_wetland,pct_open_water,pct_row_crops
0,2015,102300070209,0.4761,0.3771,0.0064,0.0568,0.0121,0.0680,0.0016,0.0017,0.8532
1,2015,102300060602,0.0581,0.0476,0.0057,0.5563,0.0829,0.1233,0.0657,0.0599,0.1057
2,2015,102300050309,0.2735,0.2089,0.0074,0.0536,0.2285,0.2149,0.0071,0.0058,0.4824
3,2015,102300050308,0.4217,0.3011,0.0215,0.0678,0.0486,0.1282,0.0080,0.0030,0.7228
4,2015,102300040202,0.5165,0.3878,0.0032,0.0468,0.0020,0.0418,0.0015,0.0001,0.9043


## Step 2 — Fix the join key

`HUC_12` arrives with its leading zero stripped: codes in HUC region `07` (the
Upper Mississippi, which covers Iowa) were exported as 11-digit numbers. Zero-pad
back to the canonical 12 characters and rename to `huc12_code`. `year` is coerced
to a plain integer.

In [4]:
before_lengths = df["HUC_12"].str.len().value_counts().sort_index()
print("HUC_12 string lengths BEFORE padding:")
print(before_lengths.to_string())

df["huc12_code"] = df["HUC_12"].str.zfill(12)
df = df.drop(columns="HUC_12")
df["year"] = df["year"].astype(int)

after_lengths = df["huc12_code"].str.len().value_counts().sort_index()
print("\nhuc12_code string lengths AFTER padding:")
print(after_lengths.to_string())
assert (df["huc12_code"].str.len() == 12).all(), "non-12-digit HUC remains"
print(f"\nUnique watersheds: {df['huc12_code'].nunique():,}")

HUC_12 string lengths BEFORE padding:
HUC_12
12    18854

huc12_code string lengths AFTER padding:
huc12_code
12    18854

Unique watersheds: 1,714


## Step 3 — Range-validate the fractions

Every `pct_*` value is a share of watershed area, so it must lie in `[0, 1]`.
We report the observed range per column and assert no value escapes those bounds
(no negatives, nothing above 1). As a sanity check we also report the spread of
the eight mutually-exclusive cover classes' row sums, which should sit near 1.

In [5]:
print("Observed ranges:")
for c in PCT_COLS:
    print(f"  {c:<18} [{df[c].min():.4f}, {df[c].max():.4f}]")

assert (df[PCT_COLS] >= 0).all().all(), "negative fraction found"
assert (df[PCT_COLS] <= 1).all().all(), "fraction > 1 found"
assert df[PCT_COLS].isna().sum().sum() == 0, "null fraction found"

COMPONENT_COLS = [c for c in PCT_COLS if c != "pct_row_crops"]
component_sum = df[COMPONENT_COLS].sum(axis=1)
print(
    f"\nCover-class row sum (8 exclusive classes): "
    f"min={component_sum.min():.3f}  mean={component_sum.mean():.3f}  "
    f"max={component_sum.max():.3f}"
)
print("(near 1.0 as expected; pct_row_crops is a derived overlap, excluded)")

Observed ranges:
  pct_corn           [0.0015, 0.7182]
  pct_soybean        [0.0007, 0.5394]
  pct_other_crops    [0.0001, 0.4869]
  pct_developed      [0.0197, 0.8434]
  pct_forest         [0.0001, 0.6308]
  pct_pasture        [0.0066, 0.6479]
  pct_wetland        [0.0000, 0.4522]
  pct_open_water     [0.0000, 0.4674]
  pct_row_crops      [0.0023, 0.9347]

Cover-class row sum (8 exclusive classes): min=0.902  mean=1.004  max=1.253
(near 1.0 as expected; pct_row_crops is a derived overlap, excluded)


## Step 4 — De-duplicate

`(huc12_code, year)` should uniquely identify a row. Drop any exact duplicate
rows and confirm the key is unique.

In [6]:
df = df.drop_duplicates()
dup_keys = df.duplicated(subset=["huc12_code", "year"]).sum()
print(f"Duplicate (huc12_code, year) keys: {dup_keys}")
assert dup_keys == 0, "duplicate key remains"
print(f"Rows after de-duplication: {len(df):,} (from {n_raw:,})")

Duplicate (huc12_code, year) keys: 0
Rows after de-duplication: 18,854 (from 18,854)


## Step 5 — Sort & write

Reorder columns to lead with the `huc12_code` + `year` key, sort, and write the
tidy table to `02_clean`.

In [7]:
ordered = ["huc12_code", "year"] + PCT_COLS
df = df[ordered].sort_values(["huc12_code", "year"]).reset_index(drop=True)

out_path = CLEAN_DIR / "cdl-huc12-fractions-clean.csv"
df.to_csv(out_path, index=False)
print(f"Wrote {len(df):,} rows × {df.shape[1]} cols to:")
print(" ", out_path.relative_to(REPO_ROOT))
df.head()

Wrote 18,854 rows × 11 cols to:
  data/tabular/02_clean/landuse/cdl-huc12-fractions-clean.csv


,huc12_code,year,pct_corn,pct_soybean,pct_other_crops,pct_developed,pct_forest,pct_pasture,pct_wetland,pct_open_water,pct_row_crops
0,070200090101,2015,0.5396,0.3456,0.0051,0.0615,0.0009,0.0158,0.0155,0.0003,0.8852
1,070200090101,2016,0.5044,0.3823,0.0081,0.0614,0.0011,0.0271,0.0041,0.0002,0.8867
2,070200090101,2017,0.5585,0.3456,0.0017,0.0605,0.0007,0.0258,0.0058,0.0002,0.9041
3,070200090101,2018,0.5005,0.4027,0.0032,0.0600,0.0008,0.0261,0.0045,0.0001,0.9032
4,070200090101,2019,0.5405,0.3624,0.0158,0.0430,0.0013,0.0239,0.0073,0.0002,0.9029
